# spin-quant: Physics-Inspired LLM Quantization

This notebook walks through the three quantization schemes and compares them on a single GPT-2 MLP layer.

## Theoretical framing

| Scheme | Physics analogy | What's stored |
|--------|----------------|---------------|
| Scalar codebook | Discrete spin states (Ising-like alphabet) | centroid index per block |
| O(N)-style | Spin orientation + magnitude | direction index + norm level |
| RG hierarchical | Coarse + short-wavelength modes | coarse index + residual index |


In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained('gpt2')
W = model.transformer.h[0].mlp.c_fc.weight.data.clone()
print(f'Weight matrix shape: {W.shape}')
print(f'Weight stats: mean={W.mean():.4f}, std={W.std():.4f}, min={W.min():.4f}, max={W.max():.4f}')

## 1. Scalar codebook (baseline)

In [ ]:
from src.codebook import quantize_blocks, reconstruct, quantization_rmse

block_dim = 16
K = 256

centroids, labels, W_shape = quantize_blocks(W, block_dim=block_dim, K=K)
Wq = reconstruct(centroids, labels, W_shape)
rmse = quantization_rmse(W, centroids, labels)
print(f'Scalar codebook RMSE: {rmse:.6f}')
print(f'Bits per weight: {__import__("math").log2(K) / block_dim:.3f}')

## 2. O(N)-style: direction + norm

In [ ]:
from src.physics import on_quantize, on_reconstruct

state_on = on_quantize(W, block_dim=block_dim, K_dir=256, n_levels=16)
Wq_on = on_reconstruct(state_on)
rmse_on = (W.float() - Wq_on).pow(2).mean().sqrt().item()
print(f'O(N) RMSE: {rmse_on:.6f}')

# Visualize norm distribution vs quantized norms
W_blocks = W.reshape(-1, block_dim).float()
norms = W_blocks.norm(dim=1)
norm_q = state_on['norm_bins'][state_on['norm_idx']]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(norms.numpy(), bins=50, alpha=0.7, label='true norms')
axes[0].hist(norm_q.numpy(), bins=50, alpha=0.7, label='quantized norms')
axes[0].set_title('Norm distribution (O(N)-style)')
axes[0].legend()

axes[1].scatter(norms.numpy()[:500], norm_q.numpy()[:500], alpha=0.3, s=2)
axes[1].set_xlabel('true norm'); axes[1].set_ylabel('quantized norm')
axes[1].set_title('Norm quantization scatter')
plt.tight_layout()
plt.show()

## 3. RG hierarchical codebook

In [ ]:
from src.physics import rg_quantize, rg_reconstruct

state_rg = rg_quantize(W, block_dim=block_dim, K_coarse=64, K_residual=64)
Wq_rg = rg_reconstruct(state_rg)
rmse_rg = (W.float() - Wq_rg).pow(2).mean().sqrt().item()
print(f'RG (coarse+residual) RMSE: {rmse_rg:.6f}')

# Compare coarse-only vs coarse+residual
Wq_coarse_only = state_rg['coarse_centroids'][state_rg['coarse_labels']].view(W.shape)
rmse_coarse = (W.float() - Wq_coarse_only).pow(2).mean().sqrt().item()
print(f'Coarse-only RMSE:          {rmse_coarse:.6f}')
print(f'Residual correction gain:  {rmse_coarse - rmse_rg:.6f}')

## 4. RMSE comparison across schemes

In [ ]:
import math

bpw_scalar = math.log2(K) / block_dim
bpw_on = (math.log2(256) + math.log2(16)) / block_dim
bpw_rg = (math.log2(64) + math.log2(64)) / block_dim

schemes = ['Scalar', 'O(N)-style', 'RG (coarse+res)']
rmses = [rmse, rmse_on, rmse_rg]
bpws = [bpw_scalar, bpw_on, bpw_rg]

print(f'{"Scheme":20s}  {"bpw":>5}  {"RMSE":>10}')
for s, b, r in zip(schemes, bpws, rmses):
    print(f'{s:20s}  {b:5.3f}  {r:10.6f}')